In [ ]:
import os
import json
import time
from datasets import load_dataset
from openai import OpenAI
from tqdm import tqdm

In [ ]:
# Setup OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

In [ ]:
MODEL_ID = "openai/gpt-oss-20b:free"

print("Loading 100 arXiv abstracts...")
dataset = load_dataset("gfissore/arxiv-abstracts-2021", split="train")
sample_papers = dataset.select(range(100))

In [ ]:
synthetic_dataset = []

for paper in tqdm(sample_papers):
    # Prompt using the "Grad Student" persona for better semantic retrieval testing
    prompt = f"""
    ROLE: You are an expert PhD researcher reviewing the following paper.
    TITLE: {paper['title']}
    ABSTRACT: {paper['abstract']}

    TASK: Generate 2 specific search queries for a vector database:
    1. TECHNICAL: A question that can ONLY be answered by the specific technical methodology of this paper. (Hard retrieval test)
    2. INTENT: A natural query from a student looking for a mentor in this specific niche. (Semantic search test)

    OUTPUT: A strict JSON list of strings. No extra text.
    Example: ["Under what specific covariance uncertainty does the framework provide a closed-form solution?", "Who is applying distributionally robust optimization to receiver design?"]
    """

    try:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.4,  # Slightly higher for more diverse "student" phrasing
        )

        raw_output = response.choices[0].message.content.strip()

        # Clean potential markdown fluff
        if "```" in raw_output:
            raw_output = raw_output.split("```")[1].replace("json", "").strip()

        queries = json.loads(raw_output)

        for q in queries:
            synthetic_dataset.append(
                {
                    "query": q,
                    "correct_paper_id": paper["id"],
                    "original_title": paper["title"],
                }
            )

    except Exception as e:
        print(f"\nError on paper {paper['id']}: {e}")

    # OpenRouter Free Tier (April 2026) has a 20 request/min limit
    time.sleep(3.1)

In [ ]:
# Save for the Evaluation script
os.makedirs("../data", exist_ok=True)
with open("../data/synthetic_eval_set.json", "w") as f:
    json.dump(synthetic_dataset, f, indent=4)

print(f"\n✅ Created {len(synthetic_dataset)} queries. Ready for embedding evaluation!")